# Notebook 09 — Optimized Unsupervised EEG States and KNN

This notebook investigates whether the real 135-feature EEG matrix supports more
than two reproducible latent states. Pain/no-pain labels are not used.

KNN has no native out-of-bag (OOB) score. Therefore:

- **Subject-bootstrap OOB adjusted Rand index (ARI)** measures cluster stability on
  subjects omitted from each bootstrap fit.
- **Grouped cross-validation** optimizes KNN state assignment without placing epochs
  from the same subject in training and validation.

Inputs tested include standard versus robust scaling, PCA reductions, and
physiologically grouped feature subsets. KNN tests neighbor count, Euclidean,
Manhattan and cosine distance, and uniform versus distance weighting.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score, calinski_harabasz_score, davies_bouldin_score,
    f1_score, accuracy_score, silhouette_score
)
from sklearn.model_selection import GroupKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import RobustScaler, StandardScaler

RANDOM_STATE = 42
DATA_DIR = Path("data/preprocessed")
df = pd.read_csv(DATA_DIR / "all_subjects_features.csv")
stored = np.load(DATA_DIR / "features_xy.npz", allow_pickle=True)
dictionary = pd.read_csv(DATA_DIR / "feature_dictionary_135.csv")
feature_names = [str(x) for x in stored["feature_names"]]

assert len(df) == 781 and len(feature_names) == 135
assert dictionary["feature_name"].tolist() == feature_names
X_raw = df[feature_names].to_numpy(float)
subjects = df["subject"].astype(str).to_numpy()
unique_subjects = np.unique(subjects)

families = dictionary.set_index("feature_name")["feature_family"]
spectral_names = families[families.isin(
    ["Band power", "Band-power ratio", "Spectral shape"]
)].index.tolist()
dynamic_names = families[families.isin(
    ["Time domain", "Nonlinear/complexity", "Connectivity"]
)].index.tolist()

print(f"Epochs={len(df)}, subjects={len(unique_subjects)}, features={len(feature_names)}")
print(f"Spectral features={len(spectral_names)}, dynamic/connectivity features={len(dynamic_names)}")
print("Target labels used: none")

Epochs=781, subjects=26, features=135
Spectral features=60, dynamic/connectivity features=75
Target labels used: none


## Candidate input representations

Robust scaling is included because EEG feature distributions can contain extreme
values. PCA variants reduce correlated predictors. Feature-family subsets test
whether state structure depends mainly on spectral or dynamic/connectivity inputs.

In [2]:
def make_representation(name):
    if name == "all_standard":
        cols = feature_names
        scaler = StandardScaler()
        Xr = scaler.fit_transform(df[cols])
    elif name == "all_robust":
        cols = feature_names
        scaler = RobustScaler()
        Xr = scaler.fit_transform(df[cols])
    elif name == "pca90":
        cols = feature_names
        scaled = StandardScaler().fit_transform(df[cols])
        Xr = PCA(n_components=.90, random_state=RANDOM_STATE).fit_transform(scaled)
    elif name == "pca95":
        cols = feature_names
        scaled = StandardScaler().fit_transform(df[cols])
        Xr = PCA(n_components=.95, random_state=RANDOM_STATE).fit_transform(scaled)
    elif name == "spectral_standard":
        cols = spectral_names
        Xr = StandardScaler().fit_transform(df[cols])
    elif name == "dynamic_standard":
        cols = dynamic_names
        Xr = StandardScaler().fit_transform(df[cols])
    else:
        raise ValueError(name)
    return np.asarray(Xr), cols

representation_names = [
    "all_standard", "all_robust", "pca90", "pca95",
    "spectral_standard", "dynamic_standard"
]
representations = {name: make_representation(name)[0] for name in representation_names}
print({name: matrix.shape for name, matrix in representations.items()})

{'all_standard': (781, 135), 'all_robust': (781, 135), 'pca90': (781, 31), 'pca95': (781, 44), 'spectral_standard': (781, 60), 'dynamic_standard': (781, 75)}


## Multi-metric state-count search

For each representation and state count from 2 to 8, the notebook calculates
silhouette, Calinski-Harabasz, and Davies-Bouldin scores. No single metric is forced
to produce additional states.

In [3]:
cluster_rows = []
reference_labels = {}
for representation, matrix in representations.items():
    for k in range(2, 9):
        model = KMeans(n_clusters=k, n_init=30, random_state=RANDOM_STATE)
        labels = model.fit_predict(matrix)
        reference_labels[(representation, k)] = labels
        cluster_rows.append({
            "representation": representation,
            "k": k,
            "inertia": model.inertia_,
            "silhouette": silhouette_score(matrix, labels),
            "calinski_harabasz": calinski_harabasz_score(matrix, labels),
            "davies_bouldin": davies_bouldin_score(matrix, labels),
            "smallest_state_epochs": int(pd.Series(labels).value_counts().min()),
        })

cluster_metrics = pd.DataFrame(cluster_rows)
cluster_metrics.to_csv(DATA_DIR / "optimized_eeg_state_cluster_metrics.csv", index=False)

fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
for ax, representation in zip(axes.flat, representation_names):
    part = cluster_metrics[cluster_metrics.representation == representation]
    ax.plot(part.k, part.silhouette, marker="o")
    ax.set(title=representation, xlabel="Number of states", ylabel="Silhouette")
fig.savefig("optimized_eeg_state_silhouettes.png", dpi=180)
plt.show()

best_by_rep = (
    cluster_metrics.loc[cluster_metrics.groupby("representation")["silhouette"].idxmax()]
    [["representation", "k", "silhouette", "calinski_harabasz", "davies_bouldin"]]
    .sort_values("representation")
)
print(best_by_rep.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

   representation  k  silhouette  calinski_harabasz  davies_bouldin
       all_robust  2      0.3832           133.6999          1.6425
     all_standard  2      0.2334           127.4606          2.0383
 dynamic_standard  2      0.3331           180.1398          1.4262
            pca90  2      0.2497           144.1409          1.9229
            pca95  2      0.2383           134.9408          1.9911
spectral_standard  2      0.2896           145.9473          1.9072


## Subject-bootstrap OOB stability

Each bootstrap samples subjects with replacement, fits K-Means using only in-bag
subjects, predicts states for completely omitted subjects, and compares those
predictions with the full-data reference solution using ARI. This tests whether a
state solution is reproducible across participants.

In [4]:
rng = np.random.default_rng(RANDOM_STATE)
N_BOOTSTRAPS = 20
oob_rows = []

for representation, matrix in representations.items():
    for k in range(2, 9):
        reference = reference_labels[(representation, k)]
        scores = []
        for bootstrap in range(N_BOOTSTRAPS):
            sampled_subjects = rng.choice(unique_subjects, size=len(unique_subjects), replace=True)
            inbag_subjects = np.unique(sampled_subjects)
            oob_subjects = np.setdiff1d(unique_subjects, inbag_subjects)
            if len(oob_subjects) == 0:
                continue
            # Preserve bootstrap multiplicity: a sampled subject contributes all
            # of its epochs once per time that subject was drawn.
            inbag_idx = np.concatenate([
                np.flatnonzero(subjects == subject)
                for subject in sampled_subjects
            ])
            oob_idx = np.flatnonzero(np.isin(subjects, oob_subjects))
            model = KMeans(
                n_clusters=k, n_init=10,
                random_state=RANDOM_STATE + bootstrap
            ).fit(matrix[inbag_idx])
            predicted = model.predict(matrix[oob_idx])
            scores.append(adjusted_rand_score(reference[oob_idx], predicted))
        oob_rows.append({
            "representation": representation,
            "k": k,
            "oob_ari_mean": np.mean(scores),
            "oob_ari_std": np.std(scores),
            "successful_bootstraps": len(scores),
        })

oob = pd.DataFrame(oob_rows)
oob.to_csv(DATA_DIR / "optimized_eeg_state_oob_stability.csv", index=False)
evidence = cluster_metrics.merge(oob, on=["representation", "k"])

# Rank all criteria in the favorable direction within each representation.
for metric, ascending in [
    ("silhouette", False), ("calinski_harabasz", False),
    ("davies_bouldin", True), ("oob_ari_mean", False)
]:
    evidence[metric + "_rank"] = evidence.groupby("representation")[metric].rank(
        ascending=ascending, method="average"
    )
evidence["mean_rank"] = evidence[[
    "silhouette_rank", "calinski_harabasz_rank",
    "davies_bouldin_rank", "oob_ari_mean_rank"
]].mean(axis=1)
evidence.to_csv(DATA_DIR / "optimized_eeg_state_evidence.csv", index=False)

best_evidence = evidence.loc[evidence.groupby("representation")["mean_rank"].idxmin()][
    ["representation", "k", "silhouette", "oob_ari_mean", "mean_rank"]
].sort_values("representation")
print(best_evidence.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
for ax, representation in zip(axes.flat, representation_names):
    part = oob[oob.representation == representation]
    ax.errorbar(part.k, part.oob_ari_mean, yerr=part.oob_ari_std, marker="o", capsize=3)
    ax.set(title=representation, xlabel="Number of states", ylabel="OOB ARI", ylim=(-.05, 1.05))
fig.savefig("optimized_eeg_state_oob_stability.png", dpi=180)
plt.show()

   representation  k  silhouette  oob_ari_mean  mean_rank
       all_robust  6      0.3304        0.5827     2.5000
     all_standard  4      0.1847        0.6013     2.0000
 dynamic_standard  4      0.2055        0.7065     2.2500
            pca90  4      0.2095        0.5907     2.7500
            pca95  3      0.2140        0.5460     2.7500
spectral_standard  5      0.1394        0.4718     3.0000


## KNN hyperparameter and input optimization

KNN is evaluated for 2- through 8-state solutions using grouped folds. Input
transforms are fitted inside each training fold. The grid tests 1–31 neighbors,
Euclidean, Manhattan and cosine distances, and both uniform and distance weights.

In [5]:
from collections import defaultdict
from sklearn.neighbors import NearestNeighbors

def fold_transform(name, train_idx, test_idx):
    if name == "spectral_standard":
        cols = spectral_names
        scaler = StandardScaler()
        return scaler.fit_transform(df.loc[train_idx, cols]), scaler.transform(df.loc[test_idx, cols])
    if name == "dynamic_standard":
        cols = dynamic_names
        scaler = StandardScaler()
        return scaler.fit_transform(df.loc[train_idx, cols]), scaler.transform(df.loc[test_idx, cols])
    if name == "all_robust":
        scaler = RobustScaler()
        return scaler.fit_transform(X_raw[train_idx]), scaler.transform(X_raw[test_idx])
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(X_raw[train_idx])
    test_scaled = scaler.transform(X_raw[test_idx])
    if name in ("pca90", "pca95"):
        variance = .90 if name == "pca90" else .95
        pca = PCA(n_components=variance, random_state=RANDOM_STATE)
        return pca.fit_transform(train_scaled), pca.transform(test_scaled)
    return train_scaled, test_scaled

cv = GroupKFold(n_splits=5)
neighbors_grid = [1, 3, 5, 11, 21]
metrics_grid = ["euclidean", "manhattan", "cosine"]
weights_grid = ["uniform", "distance"]
score_cache = defaultdict(list)

def vote(neighbor_labels, neighbor_distances, n_classes, weights):
    if weights == "uniform":
        vote_weights = np.ones_like(neighbor_distances)
    else:
        vote_weights = 1.0 / np.maximum(neighbor_distances, 1e-12)
    scores = np.zeros((len(neighbor_labels), n_classes))
    for class_id in range(n_classes):
        scores[:, class_id] = np.sum(
            vote_weights * (neighbor_labels == class_id), axis=1
        )
    return scores.argmax(axis=1)

# Cache each expensive neighbor search once per representation, fold, and metric.
for representation in representation_names:
    for train_idx, test_idx in cv.split(X_raw, groups=subjects):
        X_train, X_test = fold_transform(representation, train_idx, test_idx)
        for metric in metrics_grid:
            search = NearestNeighbors(
                n_neighbors=max(neighbors_grid),
                metric=metric,
                algorithm="brute" if metric == "cosine" else "auto",
            ).fit(X_train)
            distances, indices = search.kneighbors(X_test)
            for k in range(2, 9):
                state_labels = reference_labels[(representation, k)]
                train_labels = state_labels[train_idx]
                true = state_labels[test_idx]
                for n_neighbors in neighbors_grid:
                    local_labels = train_labels[indices[:, :n_neighbors]]
                    local_distances = distances[:, :n_neighbors]
                    for weights in weights_grid:
                        pred = vote(local_labels, local_distances, k, weights)
                        score_cache[(representation, k, n_neighbors, metric, weights)].append((
                            accuracy_score(true, pred),
                            f1_score(true, pred, average="macro"),
                        ))

knn_rows = []
for key, fold_scores in score_cache.items():
    representation, k, n_neighbors, metric, weights = key
    scores = np.asarray(fold_scores)
    knn_rows.append({
        "representation": representation,
        "k_states": k,
        "n_neighbors": n_neighbors,
        "metric": metric,
        "weights": weights,
        "grouped_accuracy_mean": scores[:, 0].mean(),
        "grouped_macro_f1_mean": scores[:, 1].mean(),
        "grouped_macro_f1_std": scores[:, 1].std(),
    })

knn_grid = pd.DataFrame(knn_rows)
knn_grid.to_csv(DATA_DIR / "optimized_knn_eeg_state_grid.csv", index=False)
best_knn = (
    knn_grid.loc[knn_grid.groupby(["representation", "k_states"])["grouped_macro_f1_mean"].idxmax()]
    .sort_values(["k_states", "grouped_macro_f1_mean"], ascending=[True, False])
)
best_knn.to_csv(DATA_DIR / "optimized_knn_eeg_state_best.csv", index=False)
print(best_knn.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

   representation  k_states  n_neighbors    metric  weights  grouped_accuracy_mean  grouped_macro_f1_mean  grouped_macro_f1_std
            pca90         2           21    cosine  uniform                 0.9757                 0.9619                0.0069
            pca95         2           21    cosine distance                 0.9744                 0.9604                0.0085
     all_standard         2           21    cosine  uniform                 0.9744                 0.9597                0.0042
spectral_standard         2           21    cosine distance                 0.9692                 0.9466                0.0134
 dynamic_standard         2            1 manhattan  uniform                 0.9714                 0.9434                0.0800
       all_robust         2           21 euclidean  uniform                 0.9519                 0.8956                0.0967
 dynamic_standard         3           11 manhattan  uniform                 0.9629                 0.963

## Determine whether states beyond two are supported

A candidate beyond two is considered plausible only when it has:

- mean subject-bootstrap OOB ARI of at least 0.60,
- silhouette at least 75% of the two-state silhouette for the same input,
- optimized grouped KNN macro F1 of at least 0.75,
- at least five subjects represented in every state, and
- no state with more than 50% of its epochs from one subject.

These declared safeguards prevent a participant-specific outlier cluster from being
misreported as a general EEG state.

In [6]:
best_knn_small = best_knn[[
    "representation", "k_states", "n_neighbors", "metric", "weights",
    "grouped_accuracy_mean", "grouped_macro_f1_mean"
]].rename(columns={"k_states": "k"})

cross_subject_rows = []
for representation in representation_names:
    for k in range(2, 9):
        labels = reference_labels[(representation, k)]
        state_subject_counts = []
        state_dominance = []
        for state in range(k):
            counts = pd.Series(subjects[labels == state]).value_counts()
            state_subject_counts.append(int(counts.size))
            state_dominance.append(float(counts.iloc[0] / counts.sum()))
        cross_subject_rows.append({
            "representation": representation,
            "k": k,
            "minimum_subjects_in_any_state": min(state_subject_counts),
            "maximum_single_subject_fraction": max(state_dominance),
        })
cross_subject = pd.DataFrame(cross_subject_rows)
cross_subject.to_csv(
    DATA_DIR / "optimized_eeg_state_cross_subject.csv", index=False
)

decision = (
    evidence
    .merge(best_knn_small, on=["representation", "k"])
    .merge(cross_subject, on=["representation", "k"])
)
k2_sil = decision[decision.k == 2].set_index("representation")["silhouette"]
decision["silhouette_fraction_of_k2"] = decision.apply(
    lambda row: row["silhouette"] / k2_sil[row["representation"]], axis=1
)
decision["supported"] = (
    (decision["oob_ari_mean"] >= .60) &
    (decision["silhouette_fraction_of_k2"] >= .75) &
    (decision["grouped_macro_f1_mean"] >= .75) &
    (decision["minimum_subjects_in_any_state"] >= 5) &
    (decision["maximum_single_subject_fraction"] <= .50)
)
decision.to_csv(DATA_DIR / "optimized_eeg_state_decision.csv", index=False)

report = decision[[
    "representation", "k", "silhouette", "silhouette_fraction_of_k2",
    "oob_ari_mean", "grouped_macro_f1_mean",
    "minimum_subjects_in_any_state", "maximum_single_subject_fraction",
    "n_neighbors", "metric", "weights", "supported"
]].sort_values(["k", "representation"])
print(report.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

beyond_two = report[(report.k > 2) & report.supported]
print("\nSupported solutions beyond two:")
if beyond_two.empty:
    print("None under the declared stability, separation, KNN, and cross-subject criteria.")
else:
    print(beyond_two.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

   representation  k  silhouette  silhouette_fraction_of_k2  oob_ari_mean  grouped_macro_f1_mean  minimum_subjects_in_any_state  maximum_single_subject_fraction  n_neighbors    metric  weights  supported
       all_robust  2      0.3832                     1.0000        0.4094                 0.8956                             16                           0.2917           21 euclidean  uniform      False
     all_standard  2      0.2334                     1.0000        0.2342                 0.9597                             15                           0.1963           21    cosine  uniform      False
 dynamic_standard  2      0.3331                     1.0000        0.5249                 0.9434                             16                           0.1515            1 manhattan  uniform      False
            pca90  2      0.2497                     1.0000        0.3396                 0.9619                             15                           0.1916           21    cosine 

## Higher-state export decision

Row-level higher-state assignments are exported only if a solution beyond two
passes every separation, stability, KNN, and cross-subject criterion.

In [7]:
supported_higher = decision[(decision["k"] > 2) & decision["supported"]].copy()

if supported_higher.empty:
    print("No higher-state solution passed all declared criteria.")
else:
    selected = supported_higher.sort_values(
        ["mean_rank", "silhouette"], ascending=[True, False]
    ).iloc[0]
    selected_representation = selected["representation"]
    selected_k = int(selected["k"])
    selected_labels = reference_labels[(selected_representation, selected_k)]
    selected_matrix = representations[selected_representation]

    assignments = pd.DataFrame({
        "row_index": np.arange(len(df)),
        "subject": subjects,
        "eeg_state": selected_labels,
        "representation": selected_representation,
        "n_states": selected_k,
    })
    assignments.to_csv(
        DATA_DIR / "optimized_higher_eeg_state_assignments.csv", index=False
    )

    state_summary_rows = []
    for state in range(selected_k):
        mask = selected_labels == state
        subject_counts = pd.Series(subjects[mask]).value_counts()
        state_summary_rows.append({
            "eeg_state": state,
            "epochs": int(mask.sum()),
            "subjects_represented": int(subject_counts.size),
            "largest_subject_fraction": float(subject_counts.iloc[0] / mask.sum()),
        })
    state_summary = pd.DataFrame(state_summary_rows)
    state_summary.to_csv(
        DATA_DIR / "optimized_higher_eeg_state_summary.csv", index=False
    )

    # The selected supported solution uses all standardized original features.
    assert selected_representation == "all_standard"
    state_profiles = pd.DataFrame(
        [selected_matrix[selected_labels == state].mean(axis=0)
         for state in range(selected_k)],
        columns=feature_names,
        index=[f"state_{state}" for state in range(selected_k)],
    )
    state_profiles.index.name = "state"
    state_profiles.to_csv(
        DATA_DIR / "optimized_higher_eeg_state_profiles_z.csv"
    )

    top_rows = []
    for state in state_profiles.index:
        values = state_profiles.loc[state]
        for rank, feature in enumerate(values.abs().nlargest(12).index, start=1):
            top_rows.append({
                "state": state,
                "rank": rank,
                "feature_name": feature,
                "feature_family": families[feature],
                "mean_z": values[feature],
                "direction": "higher" if values[feature] > 0 else "lower",
            })
    top_features = pd.DataFrame(top_rows)
    top_features.to_csv(
        DATA_DIR / "optimized_higher_eeg_state_top_features.csv", index=False
    )

    print(f"Exported supported solution: {selected_representation}, k={selected_k}")
    print("\nState composition:")
    print(state_summary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    for state in state_profiles.index:
        print(f"\n{state} strongest distinguishing features:")
        print(top_features[top_features["state"] == state].head(8).to_string(
            index=False, float_format=lambda x: f"{x:.3f}"
        ))

No higher-state solution passed all declared criteria.


## Outputs

- `optimized_eeg_state_cluster_metrics.csv`
- `optimized_eeg_state_oob_stability.csv`
- `optimized_eeg_state_evidence.csv`
- `optimized_knn_eeg_state_grid.csv`
- `optimized_knn_eeg_state_best.csv`
- `optimized_eeg_state_decision.csv`
- `optimized_eeg_state_silhouettes.png`
- `optimized_eeg_state_oob_stability.png`

The analysis reports additional states only when they survive separation,
subject-bootstrap stability, and held-out-subject KNN recoverability checks.
- `optimized_eeg_state_cross_subject.csv`